# Anirban — NeuroMatch 2026 · Posterior Motives

**Your slide's questions:**

- Hierarchical model overview.
- Show the model produces **both** unimodal and bimodal responses.
- Use one clearly **unimodal** subject and one clearly **bimodal** subject.
- Compare what changes in the model between the two cases.
- Interpret how the model captures unimodality vs bimodality.

This notebook is runnable top-to-bottom and uses **HB-Rachel** (our hierarchical Bayesian observer) and the **Switch** (paper's switching observer). Edit `SUBJECT` / filters and re-run any cell. See the API guide cell for how to make your own calls.

In [ ]:
# ============================================================
#  SETUP  —  works in Google Colab or on a local checkout
# ============================================================
import os, sys, subprocess

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
BRANCH   = "model-verification"     # branch that carries the fitted models + API

def _find_root():
    """If we're already inside a checkout, use it (no clone needed)."""
    here = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "observers", "api.py")):
            return here
        here = os.path.dirname(here)
    return None

ROOT = _find_root()
if ROOT is None:
    # Colab path: shallow-clone the repo, then point at the hierarchical/ package.
    dest = "/content/NeuroMatch_2026_Behaviour" if os.path.isdir("/content") \
           else os.path.abspath("NeuroMatch_2026_Behaviour")
    if not os.path.isdir(dest):
        # PUBLIC repo: this just works. PRIVATE repo: replace REPO_URL with
        #   https://<TOKEN>@github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                        REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, "hierarchical")

sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from observers import api

print("repo root :", ROOT)
print("models    :", api.list_models())
print("data      :", "data/data01_direction4priors.csv  (12 subjects)")

# figures for this notebook go in a dedicated subfolder beside it
FIG_DIR = os.path.join(ROOT, "experiments", "anirban", "01_slide_notebook", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("figures  :", FIG_DIR)

## How to use the model API (read me — also for an LLM assistant)

Everything below comes from the **fitted models**, reached through one module:
`observers.api`. You never touch raw model math — you call functions.

### Where the data lives (relative to the repo's `hierarchical/` folder)
| what | path |
|---|---|
| trial-level data (all 12 subjects) | `data/data01_direction4priors.csv` |
| point fits (per model, per subject) | `results/fits/comparison/<model>/subject<N>.json` |
| cross-validation records | `results/fits/comparison_cv/<model>/subject<N>_cv.json` |
| project abstract | `docs/project_abstract.md` |

### Model keys
`'switch'` (paper's Switching observer, 9 params, non-learning) · `'hb_rachel'`
(our hierarchical Bayesian observer, 7 params, **learns** prior width online) ·
also available: `'hb_adaptive'`, `'hb_salma'`, `'recombined'`, `'basic_bayes'`.

### The API — five kinds of call
```python
from observers import api

# LOAD --------------------------------------------------------------
api.load_subject(2)                 # -> DataFrame: motion_direction, motion_coherence,
                                    #    prior_std, estimate_dir  (one subject's trials)
api.load_fitted('hb_rachel', 2)     # -> (observer, record) with fitted params
api.observed_distribution(2, direction=335, coherence=0.06, prior_std=80)
                                    # -> empirical response histogram (density over 1..360)

# INSPECT -----------------------------------------------------------
api.list_models(); api.model_info() # what exists, params, colors
api.fitted_subjects('hb_rachel')    # which subjects are fit

# PREDICT -----------------------------------------------------------
api.predict('hb_rachel', 2)         # -> (n_trials, 360) predicted distribution per trial
api.belief_trajectory('hb_rachel', 2)
                                    # -> DataFrame trial, believed_sd  (the LEARNED prior width)

# FIT / SIMULATE ----------------------------------------------------
api.fit_model('hb_rachel', 2)       # refit from scratch (slow)
api.simulate('hb_rachel', 2, seed=0)# generative: synthetic responses from the fitted model

# EVALUATE ----------------------------------------------------------
api.results_table()                 # -> tidy DataFrame: model,label,subject,k,nll,aic,bic,cv_nll
api.trial_logliks('hb_rachel', 2)   # -> per-trial log-likelihood (slice it however you like)
api.bias_variability(2)             # -> per-condition estimation bias + circular SD (Fig-3 core)
```

### Custom calls (when a helper doesn't exist)
The API returns raw numbers; **you** aggregate/plot. To reach the model object
directly:
```python
from observers.comparison.registry import build_registry, load_subject
spec = build_registry(['hb_rachel'])['hb_rachel']   # a ModelSpec
obs, rec = api.load_fitted('hb_rachel', 2)
dists = spec.predict_distributions(obs, load_subject(2))  # (n_trials, 360)
```
Condition labels for any trial-level array come from `api.load_subject(s)` —
they're **row-aligned** with `predict`, `trial_logliks`, and `belief_trajectory`.

## 1 · The hierarchical observer in one picture

HB-Rachel holds a **mixed prior** — part a peaked von Mises bump at 225° (the informed prior), part a flat uniform (the naive prior) — mixed by a confidence weight **α**. Each trial it combines that prior with the sensory likelihood (width set by coherence) and reads out an estimate. Whether the response distribution is **one peak or two** is set by how sharp the evidence is: broad evidence (low coherence) leaves the prior bump standing next to the stimulus → **bimodal**; sharp evidence (high coherence) overwhelms the prior → **unimodal**.

In [ ]:
# Pick a bimodal and a unimodal subject FROM THE OBSERVED DATA (not by hand).
# Bimodality index = min(prior-cluster mass, stimulus-cluster mass) in far,
# low-coherence trials: high when BOTH clusters are populated.
def bimodality_index(s):
    df = api.load_subject(s)
    d = df.motion_direction.values; c = df.motion_coherence.values; e = df.estimate_dir.values
    far = (np.abs((d-225+180)%360-180) >= 90) & (c <= 0.12)
    if far.sum() < 30: return np.nan
    est, stim = e[far], d[far]
    prior_mass = (np.abs((est-225+180)%360-180) <= 30).mean()
    stim_mass  = (np.abs((est-stim+180)%360-180) <= 30).mean()
    return min(prior_mass, stim_mass)

idx = pd.Series({s: bimodality_index(s) for s in range(1,13)}).sort_values(ascending=False)
BIMODAL_SUBJ  = int(idx.index[0])     # both clusters populated
UNIMODAL_SUBJ = int(idx.dropna().index[-1])  # one cluster dominates
print('bimodality index by subject:'); print(idx.round(3).to_string())
print(f'\n-> bimodal example  = subject {BIMODAL_SUBJ}')
print(f'-> unimodal example = subject {UNIMODAL_SUBJ}')

## 2 · Model reproduces both shapes — observed vs predicted

For each example subject we take a **far, low-coherence, wide-prior cell** (the regime where the two clusters are most separated) and overlay the observed response histogram with HB-Rachel's prediction.

In [ ]:
def far_cell(s, prior_std=80, coh_max=0.12):
    """The far stimulus direction with the most trials in this cell."""
    df = api.load_subject(s)
    d = df.motion_direction.values; c = df.motion_coherence.values; ps = df.prior_std.values
    far = (np.abs((d-225+180)%360-180) >= 90) & (c <= coh_max) & (ps == prior_std)
    return int(pd.Series(d[far]).value_counts().index[0])

DEG = np.arange(1, 361)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, s, tag in [(axes[0], BIMODAL_SUBJ, 'bimodal'), (axes[1], UNIMODAL_SUBJ, 'unimodal')]:
    stim = far_cell(s)
    obs = api.observed_distribution(s, direction=stim, coherence=0.12, prior_std=80)
    # model prediction, averaged over the matching cell
    df = api.load_subject(s)
    pred_all = api.predict('hb_rachel', s)
    m = (df.motion_direction.values==stim) & (df.motion_coherence.values<=0.12) & (df.prior_std.values==80)
    pred = pred_all[m].mean(0); pred = pred/pred.sum()
    # smooth observed for display (10-deg bins) then plot both
    ax.bar(DEG, obs, width=1.0, color='#8a8a8a', alpha=0.5, label=f'observed (n={m.sum()})')
    ax.plot(DEG, pred, color='#edae49', lw=2.2, label='HB-Rachel')
    ax.axvline(225, ls=':', c='k', lw=1, alpha=0.6); ax.axvline(stim, ls='--', c='#c33', lw=1, alpha=0.7)
    ax.set_title(f'subject {s} — {tag}\nstim={stim}°  (prior at 225°)')
    ax.set_xlabel('estimated direction (deg)'); ax.set_xlim(0,360); ax.legend(fontsize=8)
axes[0].set_ylabel('response density')
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'anirban_bimodality.png'), dpi=130); plt.show()

## 3 · What changes in the model between the two cases

Same model, same prior structure — only the **fitted sensory reliability** differs. The confidence weight α is nearly identical; what flips the shape is how sharp the evidence is.

In [ ]:
import json
rows = []
for s, tag in [(BIMODAL_SUBJ,'bimodal'), (UNIMODAL_SUBJ,'unimodal')]:
    p = json.load(open(f'results/fits/comparison/hb_rachel/subject{s}.json'))['params']
    rows.append({'subject': s, 'case': tag, 'alpha (prior confidence)': round(p['alpha'],3),
                 'lam (forgetting)': round(p['lam'],3),
                 'k_like@0.06': round(p['k_like']['0.06'],2),
                 'k_like@0.12': round(p['k_like']['0.12'],2),
                 'k_like@0.24': round(p['k_like']['0.24'],2)})
tab = pd.DataFrame(rows); print(tab.to_string(index=False))
print('\nRead-out: alpha (prior weight) is ~equal across the two subjects.')
print('The unimodal subject has MUCH higher sensory k_like (sharp evidence),')
print('so the likelihood overwhelms the prior bump -> single peak at the stimulus.')
print('The bimodal subject has broad evidence -> prior bump survives -> two peaks.')

**Interpretation (abstract claim i).** HB-Rachel reproduces *both* shapes with a single graded mechanism — no discrete switch. Bimodality emerges when broad sensory evidence cannot overrule the mixed prior's peak, so mass sits at *both* the stimulus and 225°; unimodality emerges when sharp evidence dominates. The distinction is carried by fitted sensory reliability (k_like), while the prior-confidence weight α stays essentially fixed across subjects — evidence that the observer *grades* its reliance on the prior rather than flipping a switch.